# 07 — Risk appetite statement + cascaded limit framework

**Plain English:** The metrics in notebooks 02–06 tell us *what the book is
doing*; this notebook says *what we will tolerate*. We read a small **risk
appetite table** from `config/risk_appetite.yaml` — an amber and a red limit, an
owner, a breach action and a review cycle for each metric — then score the
current book **green / amber / red (RAG)** against those limits. Monitoring
without limits is just reporting (APS 220 para 20).

**One-line terms**
- **Risk appetite** — how much risk the Board is willing to run; expressed as limits.
- **Amber / Red limit** — early-warning level / hard limit; a red breach escalates to the Board.
- **RAG status** — green within appetite · amber approaching the limit · red breached.
- **Leading vs lagging** — leading indicators (SICR share, roll rates) move *before* defaults; lagging ones (NPL) confirm after (APG 220 para 66).
- **Cascade** — appetite flows Board → portfolio → segment → facility (APS 220 para 35).

> Illustrative demo thresholds — **not a regulatory submission**. Levels are set
> to plausible mortgage values, not fitted to this crisis+calm sample.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")
cfg = m.load_appetite()
print("appetite metrics:", list(cfg["metrics"]))

In [ ]:
# Current vs prior reporting month, all appetite metrics as-of each --------
periods = sorted(int(p) for p in panel["period"].unique())
this_period = periods[-1]
def _prev(p):
    y, mn = divmod(p, 100)
    return (y - 1) * 100 + 12 if mn == 1 else p - 1
last_period = _prev(this_period) if _prev(this_period) in set(periods) else periods[-2]

this_vals = m.portfolio_metrics_asof(panel, this_period)
last_vals = m.portfolio_metrics_asof(panel, last_period)
appetite = m.evaluate_appetite(cfg, this_vals, last_vals)
appetite.to_csv(m.OUT_TABLES / "07_appetite_status.csv", index=False)
print(f"this period {this_period} vs last period {last_period}")
appetite[["metric", "type", "last_period", "this_period", "amber", "red (limit)", "RAG"]]

In [ ]:
# Leading-indicator TREND over time (not just the pooled matrix) ----------
# APG 220 para 66: a prudent ADI uses forward-looking indicators. We track the
# trailing-12m roll rates and the SICR (Current->Stage 2) migration rate at each
# year-end, so the trend is visible, not just a single pooled number.
def sicr_rate(trans, end_period, months=12):
    end = m._period_ord(end_period)
    sub = trans[(trans["bucket"] == "Current") & trans["next_stage"].notna()].copy()
    o = m._period_ord(sub["period"])
    win = sub[(o <= end) & (o > end - months)]
    return float(100 * (win["next_stage"] == "Stage 2").mean()) if len(win) else np.nan

anchors = [p for p in periods if p % 100 == 12][-5:] or periods[-5:]
if this_period not in anchors:
    anchors = anchors + [this_period]
trend = pd.DataFrame([{
    "as_of": p,
    "roll_current_30 (leading)": round(m.roll_window(panel, p)["roll_current_30"], 3),
    "roll_30_60 (leading)": round(m.roll_window(panel, p)["roll_30_60"], 2),
    "sicr_current_to_stage2 (leading)": round(sicr_rate(panel, p), 3),
} for p in anchors])
trend.to_csv(m.OUT_TABLES / "07_leading_trends.csv", index=False)
trend

In [ ]:
# Stress -> limits: would a downturn breach the limits? (MON-7) ----------
# APS 220 para 73 / APG 220 para 76. Reuse the sister model's crisis severity
# (this repo's own vintage curves: 2007 ~4x 2015 default at 72m) as a downturn
# multiplier, and re-test the stressed metrics against their red limits.
mult = cfg["stress"]["downturn_multiplier"]
rows = []
for k in cfg["stress"]["applies_to"]:
    c = cfg["metrics"][k]
    cur = this_vals.get(k, float("nan"))
    stressed = cur * mult
    rows.append({
        "metric": c["label"],
        "current": round(cur, 2),
        f"stressed (x{mult:g})": round(stressed, 2),
        "red (limit)": c["red"],
        "RAG current": m.rag(cur, c["amber"], c["red"]),
        "RAG under stress": m.rag(stressed, c["amber"], c["red"]),
    })
stress = pd.DataFrame(rows)
stress.to_csv(m.OUT_TABLES / "07_stress_vs_limits.csv", index=False)
print(f"downturn multiplier x{mult:g} — {cfg['stress']['note']}")
stress